<a href="https://colab.research.google.com/github/Alamsyah-leh/PROKOM/blob/main/Prokom_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard Spatio-Temporal Keramaian FT Undip</title>

    <!-- Leaflet CSS -->
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>

    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
        body { display: flex; height: 100vh; background-color: #f4f6f9; }

        /* Panel Kiri (Pendekatan Sistem & Pengambilan Keputusan) */
        #sidebar { width: 350px; background: #fff; padding: 20px; box-shadow: 2px 0 5px rgba(0,0,0,0.1); z-index: 1000; overflow-y: auto; }
        h2 { color: #2c3e50; font-size: 1.2rem; margin-bottom: 20px; border-bottom: 2px solid #3498db; padding-bottom: 10px; }

        .control-group { margin-bottom: 15px; }
        label { display: block; font-size: 0.9rem; font-weight: bold; margin-bottom: 5px; color: #34495e; }
        select { width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px; outline: none; }

        /* Box Rekomendasi (Decision Making) */
        .decision-box { background: #e8f4f8; border-left: 4px solid #3498db; padding: 15px; margin-top: 20px; border-radius: 4px; }
        .decision-box h3 { font-size: 1rem; color: #2980b9; margin-bottom: 10px; }
        .decision-box p { font-size: 0.85rem; color: #333; line-height: 1.5; }

        /* Peta Kanan (Pendekatan Spasial, Activity Mapping, Density) */
        #map { flex-grow: 1; height: 100%; }

        .legend { background: white; padding: 10px; line-height: 1.5; color: #333; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2); }
        .legend i { width: 18px; height: 18px; float: left; margin-right: 8px; border-radius: 50%; }
    </style>
</head>
<body>

    <!-- SIDEBAR: Pendekatan Sistem (Dashboard UI) -->
    <div id="sidebar">
        <h2>Dashboard Keramaian FT Undip</h2>

        <!-- Pendekatan Temporal -->
        <div class="control-group">
            <label for="timeFilter">Waktu (Spatio-Temporal):</label>
            <select id="timeFilter" onchange="updateDashboard()">
                <option value="semua">Sepanjang Hari</option>
                <option value="pagi">Pagi (07.00 - 10.00)</option>
                <option value="siang">Siang (11.00 - 14.00)</option>
                <option value="sore">Sore (15.00 - 18.00)</option>
            </select>
        </div>

        <!-- Pendekatan Activity Mapping -->
        <div class="control-group">
            <label for="activityFilter">Jenis Aktivitas:</label>
            <select id="activityFilter" onchange="updateDashboard()">
                <option value="semua">Semua Aktivitas</option>
                <option value="belajar">Belajar / Kelas</option>
                <option value="makan">Makan / Kantin</option>
                <option value="nongkrong">Nongkrong / Diskusi</option>
                <option value="transit">Transit / Halte</option>
            </select>
        </div>

        <!-- Pemilihan Analisis -->
        <div class="control-group">
            <label for="viewMode">Metode Analisis:</label>
            <select id="viewMode" onchange="updateDashboard()">
                <option value="heatmap">Analisis Kepadatan (Heatmap)</option>
                <option value="marker">Activity Mapping (Titik Fungsi)</option>
            </select>
        </div>

        <!-- Pendekatan Pengambilan Keputusan (Decision Support) -->
        <div class="decision-box">
            <h3>Rekomendasi Berbasis Data</h3>
            <p id="decisionText">Pilih filter untuk melihat rekomendasi pengaturan fasilitas dan jadwal berdasarkan data keramaian terkini.</p>
        </div>
    </div>

    <!-- WADAH PETA -->
    <div id="map"></div>

    <!-- Leaflet & Heatmap JS -->
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://unpkg.com/leaflet.heat@0.2.0/dist/leaflet-heat.js"></script>

    <script>
        // 1. PENDEKATAN SPASIAL: Inisialisasi Peta
        // Koordinat diset di area Fakultas Teknik Universitas Diponegoro
        const map = L.map('map').setView([-7.0515, 110.4377], 17);

        // Tambahkan Basemap dari OpenStreetMap
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
            attribution: '© OpenStreetMap contributors',
            maxZoom: 19
        }).addTo(map);

        // 2. DATA AKTIVITAS (Simulasi Data Pengumpulan Otomatis)
        const dummyData = [
            // Pagi - Aktivitas didominasi belajar di kelas / Transit
            { lat: -7.0518, lng: 110.4370, type: 'belajar', time: 'pagi', weight: 0.9, name: 'Gedung Departemen Teknik Geodesi' },
            { lat: -7.0512, lng: 110.4382, type: 'belajar', time: 'pagi', weight: 0.8, name: 'Gedung Kuliah Bersama FT' },
            { lat: -7.0505, lng: 110.4375, type: 'transit', time: 'pagi', weight: 0.9, name: 'Halte Teknik' },
            { lat: -7.0520, lng: 110.4365, type: 'transit', time: 'pagi', weight: 0.7, name: 'Area Parkir Sipil' },

            // Siang - Aktivitas didominasi makan dan transit
            { lat: -7.0525, lng: 110.4380, type: 'makan', time: 'siang', weight: 1.0, name: 'Kantin Teknik (G-KM)' },
            { lat: -7.0518, lng: 110.4370, type: 'nongkrong', time: 'siang', weight: 0.8, name: 'Lobi Teknik Geodesi' },
            { lat: -7.0508, lng: 110.4368, type: 'makan', time: 'siang', weight: 0.9, name: 'Kantin Arsitektur' },
            { lat: -7.0515, lng: 110.4390, type: 'belajar', time: 'siang', weight: 0.6, name: 'Perpustakaan FT' },

            // Sore - Aktivitas didominasi nongkrong, organisasi (Litbang/Himpunan), dan transit pulang
            { lat: -7.0522, lng: 110.4373, type: 'nongkrong', time: 'sore', weight: 0.8, name: 'Sekretariat Himpunan' },
            { lat: -7.0518, lng: 110.4370, type: 'belajar', time: 'sore', weight: 0.5, name: 'Lab Komputer Geodesi' },
            { lat: -7.0505, lng: 110.4375, type: 'transit', time: 'sore', weight: 1.0, name: 'Halte Teknik (Antrean Trans Jateng)' },
            { lat: -7.0525, lng: 110.4380, type: 'nongkrong', time: 'sore', weight: 0.7, name: 'Kantin Teknik' },

            // Beberapa titik tambahan acak untuk memperjelas heatmap
            { lat: -7.0519, lng: 110.4371, type: 'belajar', time: 'pagi', weight: 0.6, name: 'Ruang Kelas' },
            { lat: -7.0524, lng: 110.4379, type: 'makan', time: 'siang', weight: 0.9, name: 'Area Nasi Uduk' },
            { lat: -7.0506, lng: 110.4376, type: 'transit', time: 'sore', weight: 0.8, name: 'Trotoar Halte' }
        ];

        // Layer Group untuk marker dan heatmap
        let heatLayer = L.heatLayer([], { radius: 25, blur: 15, maxZoom: 17 }).addTo(map);
        let markerLayer = L.layerGroup().addTo(map);

        // Warna custom untuk Activity Mapping
        const getColor = (type) => {
            return type === 'belajar' ? '#3498db' : // Biru
                   type === 'makan' ? '#e74c3c' :   // Merah
                   type === 'nongkrong' ? '#f1c40f' : // Kuning
                   '#9b59b6'; // Ungu (Transit)
        };

        // Fungsi utama untuk mengupdate peta dan data
        function updateDashboard() {
            const timeFilter = document.getElementById('timeFilter').value;
            const activityFilter = document.getElementById('activityFilter').value;
            const viewMode = document.getElementById('viewMode').value;

            // Filter data berdasarkan Waktu dan Aktivitas
            const filteredData = dummyData.filter(d =>
                (timeFilter === 'semua' || d.time === timeFilter) &&
                (activityFilter === 'semua' || d.type === activityFilter)
            );

            // Bersihkan layer sebelumnya
            map.removeLayer(heatLayer);
            markerLayer.clearLayers();

            if (viewMode === 'heatmap') {
                // 3. PENDEKATAN ANALISIS KEPADATAN (KDE / Heatmap)
                const heatPoints = filteredData.map(d => [d.lat, d.lng, d.weight]);
                heatLayer = L.heatLayer(heatPoints, { radius: 30, blur: 20, maxZoom: 17, max: 1.0 }).addTo(map);
            } else {
                // 2. PENDEKATAN ACTIVITY MAPPING (Klasifikasi Fungsi)
                filteredData.forEach(d => {
                    L.circleMarker([d.lat, d.lng], {
                        radius: 8,
                        fillColor: getColor(d.type),
                        color: "#fff",
                        weight: 1,
                        opacity: 1,
                        fillOpacity: 0.8
                    })
                    .bindPopup(`<b>${d.name}</b><br>Aktivitas: ${d.type.toUpperCase()}<br>Waktu: ${d.time.toUpperCase()}`)
                    .addTo(markerLayer);
                });
            }

            // 6. PENDEKATAN PENGAMBILAN KEPUTUSAN (Update Teks Rekomendasi)
            updateRecommendations(timeFilter, activityFilter, filteredData.length);
        }

        // Fungsi Decision Support System
        function updateRecommendations(time, activity, count) {
            const textElement = document.getElementById('decisionText');
            let rekomendasi = "";

            if (count === 0) {
                rekomendasi = "Tidak ada kepadatan signifikan terdeteksi untuk parameter ini.";
            } else if (time === 'pagi') {
                rekomendasi = "<b>Kepadatan:</b> Menumpuk di area parkir dan Halte transit.<br><br><b>Aksi:</b> Perlu penambahan personel satpam untuk mengurai kemacetan di gerbang utama FT dan pengaturan titik jemput ojek online.";
            } else if (time === 'siang') {
                rekomendasi = "<b>Kepadatan:</b> Terpusat di fasilitas makanan (Kantin Teknik, Kantin Arsitektur).<br><br><b>Aksi:</b> Disarankan redistribusi ruang makan mahasiswa atau penambahan *food truck* sementara untuk mengurangi penumpukan antrean.";
            } else if (time === 'sore') {
                rekomendasi = "<b>Kepadatan:</b> Menumpuk di Halte Teknik dan area sekretariat himpunan.<br><br><b>Aksi:</b> Pengajuan penambahan frekuensi bus Trans Jateng pada jam pulang (16.00-17.00).";
            } else {
                rekomendasi = "Menampilkan data agregat. Analisis spasial secara keseluruhan menunjukkan perlunya pemisahan zona akademik dan zona transit di sekitar lingkar Fakultas Teknik.";
            }

            textElement.innerHTML = rekomendasi;
        }

        // Menambahkan Legenda untuk mode Marker
        const legend = L.control({position: 'bottomright'});
        legend.onAdd = function (map) {
            const div = L.DomUtil.create('div', 'legend');
            div.innerHTML += '<strong>Legenda Aktivitas</strong><br>';
            div.innerHTML += '<i style="background: #3498db"></i> Belajar<br>';
            div.innerHTML += '<i style="background: #e74c3c"></i> Makan<br>';
            div.innerHTML += '<i style="background: #f1c40f"></i> Nongkrong<br>';
            div.innerHTML += '<i style="background: #9b59b6"></i> Transit<br>';
            return div;
        };
        legend.addTo(map);

        // Inisialisasi awal saat load
        updateDashboard();

    </script>
</body>
</html>

SyntaxError: invalid decimal literal (952980898.py, line 13)

In [ ]:
from IPython.display import HTML

html_code = """
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard Keramaian FT Undip</title>

    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>

    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', Tahoma, sans-serif; }

        /* Layout diubah menjadi Kolom (Atas-Bawah) agar ramah layar HP */
        body { display: flex; flex-direction: column; background-color: #f4f6f9; }

        /* Panel Atas mengambil lebar penuh */
        #sidebar { width: 100%; background: #fff; padding: 15px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); z-index: 1000; }
        h2 { font-size: 1.1rem; margin-bottom: 10px; border-bottom: 2px solid #3498db; padding-bottom: 5px; color: #2c3e50; }

        .controls-container { display: flex; flex-wrap: wrap; gap: 10px; }
        .control-group { flex: 1; min-width: 130px; }
        label { display: block; font-size: 0.8rem; font-weight: bold; margin-bottom: 4px; color: #34495e; }
        select { width: 100%; padding: 6px; border: 1px solid #ccc; border-radius: 4px; font-size: 0.85rem; }

        .decision-box { background: #e8f4f8; border-left: 4px solid #3498db; padding: 10px; margin-top: 10px; border-radius: 4px; }
        .decision-box h3 { font-size: 0.9rem; color: #2980b9; margin-bottom: 5px; }
        .decision-box p { font-size: 0.8rem; color: #333; line-height: 1.4; }

        /* Peta dipaksa memiliki tinggi absolut (450px) agar tidak hilang di Colab */
        #map { width: 100%; height: 450px; z-index: 1; }

        .legend { background: white; padding: 8px; font-size: 0.8rem; line-height: 1.5; color: #333; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2); }
        .legend i { width: 12px; height: 12px; float: left; margin-right: 5px; border-radius: 50%; }
    </style>
</head>
<body>

    <div id="sidebar">
        <h2>Dashboard Keramaian FT Undip</h2>

        <div class="controls-container">
            <div class="control-group">
                <label>Waktu (Spatio-Temporal):</label>
                <select id="timeFilter" onchange="updateDashboard()">
                    <option value="semua">Sepanjang Hari</option>
                    <option value="pagi">Pagi (07.00 - 10.00)</option>
                    <option value="siang">Siang (11.00 - 14.00)</option>
                    <option value="sore">Sore (15.00 - 18.00)</option>
                </select>
            </div>

            <div class="control-group">
                <label>Aktivitas:</label>
                <select id="activityFilter" onchange="updateDashboard()">
                    <option value="semua">Semua Aktivitas</option>
                    <option value="belajar">Belajar / Kelas</option>
                    <option value="makan">Makan / Kantin</option>
                    <option value="nongkrong">Nongkrong</option>
                    <option value="transit">Transit / Halte</option>
                </select>
            </div>

            <div class="control-group">
                <label>Metode Analisis:</label>
                <select id="viewMode" onchange="updateDashboard()">
                    <option value="heatmap">Heatmap (Kepadatan)</option>
                    <option value="marker">Titik Mapping</option>
                </select>
            </div>
        </div>

        <div class="decision-box">
            <h3>Rekomendasi Keputusan</h3>
            <p id="decisionText">Pilih filter untuk melihat rekomendasi pengaturan fasilitas.</p>
        </div>
    </div>

    <div id="map"></div>

    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://unpkg.com/leaflet.heat@0.2.0/dist/leaflet-heat.js"></script>

    <script>
        const map = L.map('map').setView([-7.0515, 110.4377], 17);

        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
            attribution: '© OSM',
            maxZoom: 19
        }).addTo(map);

        const dummyData = [
            { lat: -7.0518, lng: 110.4370, type: 'belajar', time: 'pagi', weight: 0.9, name: 'Gedung Geodesi' },
            { lat: -7.0512, lng: 110.4382, type: 'belajar', time: 'pagi', weight: 0.8, name: 'GKB FT' },
            { lat: -7.0505, lng: 110.4375, type: 'transit', time: 'pagi', weight: 0.9, name: 'Halte Teknik' },
            { lat: -7.0520, lng: 110.4365, type: 'transit', time: 'pagi', weight: 0.7, name: 'Parkir Sipil' },
            { lat: -7.0525, lng: 110.4380, type: 'makan', time: 'siang', weight: 1.0, name: 'Kantin Teknik' },
            { lat: -7.0518, lng: 110.4370, type: 'nongkrong', time: 'siang', weight: 0.8, name: 'Lobi Geodesi' },
            { lat: -7.0508, lng: 110.4368, type: 'makan', time: 'siang', weight: 0.9, name: 'Kantin Arsitektur' },
            { lat: -7.0515, lng: 110.4390, type: 'belajar', time: 'siang', weight: 0.6, name: 'Perpus FT' },
            { lat: -7.0522, lng: 110.4373, type: 'nongkrong', time: 'sore', weight: 0.8, name: 'Sekret Himpunan' },
            { lat: -7.0518, lng: 110.4370, type: 'belajar', time: 'sore', weight: 0.5, name: 'Lab Komputer' },
            { lat: -7.0505, lng: 110.4375, type: 'transit', time: 'sore', weight: 1.0, name: 'Halte Teknik' },
            { lat: -7.0525, lng: 110.4380, type: 'nongkrong', time: 'sore', weight: 0.7, name: 'Kantin Teknik' }
        ];

        let heatLayer = L.heatLayer([], { radius: 25, blur: 15, maxZoom: 17 }).addTo(map);
        let markerLayer = L.layerGroup().addTo(map);

        const getColor = (type) => {
            return type === 'belajar' ? '#3498db' :
                   type === 'makan' ? '#e74c3c' :
                   type === 'nongkrong' ? '#f1c40f' : '#9b59b6';
        };

        function updateDashboard() {
            const timeFilter = document.getElementById('timeFilter').value;
            const activityFilter = document.getElementById('activityFilter').value;
            const viewMode = document.getElementById('viewMode').value;

            const filteredData = dummyData.filter(d =>
                (timeFilter === 'semua' || d.time === timeFilter) &&
                (activityFilter === 'semua' || d.type === activityFilter)
            );

            map.removeLayer(heatLayer);
            markerLayer.clearLayers();

            if (viewMode === 'heatmap') {
                const heatPoints = filteredData.map(d => [d.lat, d.lng, d.weight]);
                heatLayer = L.heatLayer(heatPoints, { radius: 30, blur: 20, maxZoom: 17, max: 1.0 }).addTo(map);
            } else {
                filteredData.forEach(d => {
                    L.circleMarker([d.lat, d.lng], {
                        radius: 8, fillColor: getColor(d.type), color: "#fff", weight: 1, opacity: 1, fillOpacity: 0.8
                    })
                    .bindPopup(`<b>${d.name}</b><br>Aktivitas: ${d.type.toUpperCase()}<br>Waktu: ${d.time.toUpperCase()}`)
                    .addTo(markerLayer);
                });
            }

            updateRecommendations(timeFilter, activityFilter, filteredData.length);
        }

        function updateRecommendations(time, activity, count) {
            const textElement = document.getElementById('decisionText');
            if (count === 0) {
                textElement.innerHTML = "Tidak ada data keramaian dengan filter ini.";
            } else if (time === 'pagi') {
                textElement.innerHTML = "<b>Kepadatan:</b> Menumpuk di parkiran & halte.<br><b>Aksi:</b> Penambahan personel keamanan di gerbang.";
            } else if (time === 'siang') {
                textElement.innerHTML = "<b>Kepadatan:</b> Terpusat di kantin.<br><b>Aksi:</b> Redistribusi ruang makan atau tambah food truck.";
            } else if (time === 'sore') {
                textElement.innerHTML = "<b>Kepadatan:</b> Menumpuk di area transit.<br><b>Aksi:</b> Pengajuan frekuensi bus Trans Jateng sore hari.";
            } else {
                textElement.innerHTML = "Menampilkan seluruh sebaran aktivitas di Fakultas Teknik.";
            }
        }

        const legend = L.control({position: 'bottomright'});
        legend.onAdd = function (map) {
            const div = L.DomUtil.create('div', 'legend');
            div.innerHTML += '<strong>Legenda</strong><br>';
            div.innerHTML += '<i style="background: #3498db"></i> Belajar<br>';
            div.innerHTML += '<i style="background: #e74c3c"></i> Makan<br>';
            div.innerHTML += '<i style="background: #f1c40f"></i> Nongkrong<br>';
            div.innerHTML += '<i style="background: #9b59b6"></i> Transit<br>';
            return div;
        };
        legend.addTo(map);

        updateDashboard();

        // FIX UNTUK COLAB: Paksa peta merender ulang ukurannya setelah dimuat
        setTimeout(() => { map.invalidateSize(); }, 500);

    </script>
</body>
</html>
"""

# Tampilkan di Colab
HTML(html_code)